# Explore the stored results

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/chris-dare-dev/k3-derived-isogenies/HEAD?labpath=research%2Fnotebooks%2F01-explore-the-results.ipynb)

This notebook reads the stored results of the scripts in `experiments/` and recomputes the headline numbers of
[SUMMARY.md](../../SUMMARY.md) from them. Nothing here computes a surface: every cell reads files, so the whole
notebook runs in a few seconds. To run small instances live, open
[02-small-live-instances.ipynb](02-small-live-instances.ipynb).

In [ ]:
import json, math, statistics
from k3data import ensure_data, run, EXP
assert ensure_data(), "stored results missing; see data/README.md"
def load(name):
    with open(EXP / name) as fh:
        return [json.loads(line) for line in fh if line.strip()]

## 1. The ledger

`sixlines_ledger.py` ([note 27](../docs/27-hypothesis-ledger.md)) compares numbers quoted in the notes, as first
written, with the same numbers recounted from the stored files. The eight that differ are the slips that the notes
now state corrected. The run should reproduce the stored log exactly.

In [ ]:
out = run("sixlines_ledger.py")
stored = (EXP / "sixlines_ledger.log").read_text()
print("identical to the stored log:", out == stored)
print("\n".join(out.splitlines()[-14:]))

## 2. The size of the group

For a maximal order the group acting on the surfaces is $G = \mathrm{Cl}(E)/\langle \mathrm{Cl}(E_0), \text{ramified primes}\rangle$
([note 13](../docs/13-hardness-on-paper.md) §3). The 512 groups of note 13 are the maximal records of the exhaustive
runs for $13 \le q \le 101$ and of the samples for $q = 211, \dots, 100{,}003$.

In [ ]:
gs = load("sixlines_groupsize.jsonl")
first = {13,17,19,23,25,29,31,37,41,43,47,49,53,59,61,67,71,73,79,83,89,97,101}
sample = {211,503,1009,2003,5003,10007,30011,100003}
groups = [r for r in gs if r["maximal"] and "G" in r and
          ((r["source"] == "all" and r["q"] in first) or (r["source"] == "sample" and r["q"] in sample))]
ratios = [r["G"] / r["q"]**2 for r in groups]
q1, med, q3 = statistics.quantiles(ratios, n=4, method="inclusive")
def slope(rs):
    xs = [math.log(r["q"]) for r in rs]; ys = [math.log(r["G"]) for r in rs]
    mx, my = sum(xs)/len(xs), sum(ys)/len(ys)
    b = sum((x-mx)*(y-my) for x, y in zip(xs, ys)) / sum((x-mx)**2 for x in xs)
    return b, math.exp(my - b*mx)
b41, _ = slope([r for r in groups if r["q"] >= 41])
b211, c211 = slope([r for r in groups if r["q"] >= 211])
print(f"{len(groups)} groups; |G|/q^2: median {med:.4f}, quartiles {q1:.4f} and {q3:.4f}")
print(f"fit log|G| = b log q + c: b = {b41:.2f} (q >= 41), b = {b211:.2f} with constant {c211:.3f} (q >= 211)")
print(f"q needed for |G| = 2^256 at |G| = 0.01 q^2: about 2^{(256 - math.log2(0.01)) / 2:.1f}")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
if plt:
    qs = [r["q"] for r in groups]; G = [r["G"] for r in groups]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.loglog(qs, G, ".", alpha=0.5, label="maximal orders")
    xs = [10, 2e5]
    ax.loglog(xs, [0.01 * x**2 for x in xs], "-", label="0.01 q²")
    ax.set_xlabel("q"); ax.set_ylabel("|G|"); ax.legend(); ax.set_title("Size of the group acting on six-line surfaces")
    plt.show()

## 3. The leaked bit: δ against the cosets of ker χ₋₁

At $q = 41$ ([note 26](../docs/26-delta-is-the-character-at-q-41.md)) the sign δ, computed on the fourfold, is compared
with three independent computations of the cosets of the kernel of $\chi_{-1}$.

In [ ]:
cmp = json.loads((EXP / "sixlines_delta41_compare.json").read_text())
for e1, rec in cmp.items():
    if not isinstance(rec, dict) or "surfaces" not in rec:
        continue
    print(f"q = 41, e1 = {e1}: {rec['surfaces']} surfaces, δ constant on all pairs of lines for {rec['delta_constant']}")
    for method, r in rec.items():
        if isinstance(r, dict) and "agree" in r:
            print(f"   {method:22s} cosets {r['sizes']}, δ by coset {r['delta_by_coset']}, agree {r['agree']}")

## 4. The leakage audit

[Note 28](../docs/28-leakage-audit.md) tests about 11,400 candidate functions of the surfaces against the labels. The
smallest p-values, with the Holm correction over each population:

In [ ]:
aud = load("sixlines_leakage.jsonl")
names = {"L": "full labels, 9 torsors", "C": "cosets of ker χ₋₁, 5 torsors"}
for pop in ("L", "C"):
    rs = sorted((r for r in aud if r["pop"] == pop), key=lambda r: r["p"])
    print(f"population {pop} ({names[pop]}): {len(rs)} candidates")
    for r in rs[:5]:
        print(f"   p = {r['p']:.3g}   Holm = {r['holm']:.3g}   {r['name'][:80]}")